# Week 7 Exercise — Fine-tuning data & evaluation for Africa / Côte d'Ivoire

**Author:** asket  
**Location:** `community_contributions/asket/`

**Not Week 5 (RAG):** Week 5 = retrieval + chat over a knowledge base (Q&A). **Week 7 = fine-tuning** a model for **price prediction**: we prepare **prompt + completion** training data (XOF), export it for Colab/QLoRA, then **evaluate** baseline vs LLM and compare errors.

This notebook prepares **training-ready data** (Day 2) and runs **evaluation** (Day 5) for price prediction in **XOF** (Côte d'Ivoire / West Africa). Use case: **market price transparency**; data can be used to fine-tune an open-source model (e.g. Llama + QLoRA in Colab) for local, low-cost inference.

---

## Week 7 roadmap

| Day | Topic |
|-----|--------|
| Day 1 | QLoRA |
| Day 2 | Prompt data and base model |
| Day 3–4 | Train (fine-tuning) |
| Day 5 | Eval |

Here: **Day 2** (prompt/completion format, export for training), **in-notebook open-source fine-tuning** (DistilGPT2 on XOF data), and **Day 5** (eval: baseline vs LLM API vs fine-tuned model).

## Setup — Imports and config

**Dependencies:** openai, python-dotenv, gradio, pandas, plotly, scikit-learn (or use notebook install cell).

In [7]:
# Install (run once if needed); on Homebrew Python use --user --break-system-packages
# %pip install -q openai python-dotenv gradio pandas plotly scikit-learn --user --break-system-packages

In [8]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import mean_absolute_error, r2_score

load_dotenv(override=True)

MODEL = "gpt-4o-mini"
CURRENCY = "XOF"
PREFIX_XOF = "Prix en XOF "
QUESTION_FR = "Quel est le prix de ce produit en francs CFA (XOF) ? Réponds par un seul nombre."
QUESTION_EN = "What is the price of this product in West African CFA francs (XOF)? Reply with a single number."
DEFAULT_N_EVAL = 15

In [9]:
if os.environ.get("OPENROUTER_API_KEY"):
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])
    print("Using OpenRouter")
else:
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))
    print("Using OpenAI")

Using OpenRouter


## Data — West Africa / Côte d'Ivoire (XOF)

Load product descriptions and prices in **XOF**. Same prompt format as Week 7 (question + summary + prefix + completion) but in XOF for local relevance.

In [10]:
def _parse_row(row: str):
    idx = row.rfind(",")
    if idx == -1:
        return None
    text = row[:idx].strip().strip('"')
    try:
        price = float(row[idx + 1:].strip())
    except ValueError:
        return None
    return (text, price)

def load_data_xof():
    """Load (text, price) in XOF. Tries week6/human_out.csv or week7 path; fallback = West Africa sample."""
    for path in [Path("../../human_out.csv"), Path("week6/human_out.csv"), Path("../human_out.csv")]:
        if path.exists():
            items = []
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    p = _parse_row(line)
                    if p: items.append({"text": p[0], "price": p[1]})
            if items:
                print(f"Loaded {len(items)} items from {path} (XOF)")
                return items
    # Fallback: West Africa / Côte d'Ivoire sample (XOF)
    sample = [
        ("Title: Riz 25 kg (parboiled, local)\nCategory: Alimentation\nDescription: Sac de 25 kg de riz parboiled, marché ivoirien.", 12500),
        ("Title: Lampe solaire LED 5W\nCategory: Électronique\nDescription: Lampe rechargeable solaire pour éclairage hors réseau.", 3500),
        ("Title: Huile de cuisson 1L\nCategory: Alimentation\nDescription: Bouteille d'huile végétale 1 litre.", 1200),
        ("Title: Ciment 50 kg\nCategory: BTP\nDescription: Sac de ciment Portland 50 kg.", 4500),
        ("Title: Téléphone Android d'occasion 4G\nCategory: Électronique\nDescription: Smartphone Android reconditionné, 2 Go RAM.", 35000),
        ("Title: Cartable écolier\nCategory: Accessoires\nDescription: Cartable solide pour écoliers.", 5500),
        ("Title: Moustiquaire imprégnée\nCategory: Santé\nDescription: Moustiquaire imprégnée 1 ou 2 places.", 2500),
        ("Title: Casque moto\nCategory: Automobile\nDescription: Casque de sécurité moto.", 8500),
        ("Title: Farine de maïs 1 kg\nCategory: Alimentation\nDescription: Paquet 1 kg pour bouillie.", 600),
        ("Title: Chaises plastique lot de 4\nCategory: Mobilier\nDescription: 4 chaises plastique pour intérieur/extérieur.", 18000),
        ("Title: Cacao 1 kg fèves\nCategory: Agriculture\nDescription: Fèves de cacao Côte d'Ivoire 1 kg.", 2200),
        ("Title: Savon de lessive 1 kg\nCategory: Hygiène\nDescription: Savon en barre pour lessive.", 800),
        ("Title: Pile AA lot 4\nCategory: Électronique\nDescription: 4 piles AA alcalines.", 1500),
        ("Title: Bidon eau 20L\nCategory: Alimentation\nDescription: Eau potable 20 litres.", 500),
        ("Title: Télévision LED 32 pouces\nCategory: Électronique\nDescription: TV LED 32 pouces entrée de gamme.", 95000),
    ]
    items = [{"text": t, "price": p} for t, p in sample]
    print(f"Using West Africa / Côte d'Ivoire sample: {len(items)} items (XOF)")
    return items

data = load_data_xof()

Using West Africa / Côte d'Ivoire sample: 15 items (XOF)


## Prompt format (Day 2 style)

Week 7 uses: **Question** + **summary** + **Prefix** + **completion**. For Africa we use the same structure in XOF (French or English).

In [11]:
def make_prompt_completion(item: dict, lang: str = "fr") -> tuple:
    """Return (prompt, completion) for this item. prompt = question + text + prefix; completion = price in XOF."""
    q = QUESTION_FR if lang == "fr" else QUESTION_EN
    prompt = f"{q}\n\n{item['text']}\n\n{PREFIX_XOF}"
    completion = f"{int(round(item['price']))}"
    return prompt, completion

# Example
p, c = make_prompt_completion(data[0], "fr")
print("Prompt (extract):", p[:200], "...")
print("Completion:", c)

# Day 2 style: prompt length stats (for fine-tuning chunk/context decisions)
prompts, completions = zip(*[make_prompt_completion(d, "fr") for d in data])
len_p = [len(pr) for pr in prompts]
len_c = [len(co) for co in completions]
print(f"\nPrompt length: min={min(len_p)}, max={max(len_p)}, avg={sum(len_p)/len(len_p):.0f} chars")
print(f"Completion length: min={min(len_c)}, max={max(len_c)} chars")

Prompt (extract): Quel est le prix de ce produit en francs CFA (XOF) ? Réponds par un seul nombre.

Title: Riz 25 kg (parboiled, local)
Category: Alimentation
Description: Sac de 25 kg de riz parboiled, marché ivoirien ...
Completion: 12500

Prompt length: min=172, max=215, avg=191 chars
Completion length: min=3, max=5 chars


## Training data export (for Colab / QLoRA)

Build prompt/completion pairs and export as JSONL for fine-tuning (Week 7 Day 3–4). Week 5 (RAG) does not export training data.

In [12]:
import json

train_data = [{"prompt": make_prompt_completion(d, "fr")[0], "completion": make_prompt_completion(d, "fr")[1]} for d in data]
out_path = Path("pricer_train_xof.jsonl")
with open(out_path, "w", encoding="utf-8") as f:
    for row in train_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Exported {len(train_data)} pairs to {out_path}")
display(pd.DataFrame(train_data).head(2))

Exported 15 pairs to pricer_train_xof.jsonl


,prompt,completion
0,Quel est le prix de ce produit en francs CFA (...,12500
1,Quel est le prix de ce produit en francs CFA (...,3500


## Open-source fine-tuning (in-notebook)

We **fine-tune an open-source causal LM** on the XOF prompt/completion data so the deliverable clearly includes a training step. We use **DistilGPT2** (small, no gated access); the same pattern works with larger models (e.g. Llama + QLoRA in Colab). Training runs here in the notebook; the saved model is used below as the **Fine-tuned (OS)** predictor.

In [17]:
# Run once if needed (uncomment and run this cell first if you get ModuleNotFoundError: datasets)
%pip install -q torch transformers datasets 'accelerate>=1.1.0' --user --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [18]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset

FT_MODEL_NAME = "distilgpt2"
FT_OUTPUT_DIR = Path("./pricer_xof_finetuned")  # If 'No space left on device', free disk or use e.g. Path("/tmp/pricer_xof_finetuned")
FT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Build HF Dataset: one example = full prompt + completion (causal LM next-token prediction)
def build_ft_dataset(items_prompt_completion):
    # items_prompt_completion = list of {"prompt": ..., "completion": ...}
    texts = [row["prompt"] + " " + str(row["completion"]) for row in items_prompt_completion]
    return Dataset.from_dict({"text": texts})

train_dataset = build_ft_dataset(train_data)
tokenizer_ft = AutoTokenizer.from_pretrained(FT_MODEL_NAME)
tokenizer_ft.pad_token = tokenizer_ft.eos_token

def tokenize(examples):
    return tokenizer_ft(examples["text"], truncation=True, max_length=256, padding="max_length")

train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
train_dataset.set_format("torch")

model_ft = AutoModelForCausalLM.from_pretrained(FT_MODEL_NAME)
if getattr(model_ft.config, 'loss_type', None) is None:
    model_ft.config.loss_type = 'ForCausalLMLoss'  # avoid config warning
print(f"Fine-tuning base model: {FT_MODEL_NAME} on {len(train_data)} XOF prompt/completion pairs")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 8577.77it/s]


Fine-tuning base model: distilgpt2 on 15 XOF prompt/completion pairs


In [27]:
# Plain PyTorch training loop (no accelerate dependency)
import torch
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_ft = model_ft.to(device)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
optimizer = torch.optim.AdamW(model_ft.parameters(), lr=5e-5)
num_epochs = 15
model_ft.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        optimizer.zero_grad()
        outputs = model_ft(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, loss: {total_loss/len(train_loader):.4f}")

# Save with torch.save (if disk full, we keep model in memory for the next cell)
import shutil
try:
    FT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model_ft.config.save_pretrained(str(FT_OUTPUT_DIR))
    tmp_pt = Path("/tmp") / "pricer_xof_pytorch_model.pt"
    torch.save(model_ft.state_dict(), tmp_pt, _use_new_zipfile_serialization=False)
    dest = FT_OUTPUT_DIR / "pytorch_model.pt"
    shutil.copy(str(tmp_pt), str(dest))
    tmp_pt.unlink(missing_ok=True)
    tokenizer_ft.save_pretrained(str(FT_OUTPUT_DIR))
    print(f"Saved fine-tuned model to {FT_OUTPUT_DIR}")
except (RuntimeError, OSError) as e:
    if "No space left" in str(e) or "errno 28" in str(e):
        print("Could not save (no space on device). Model kept in memory — run the next cell to use it.")
    else:
        raise
print("Done. Run the next cell to load the model for inference.")

Epoch 5/15, loss: 0.0591
Epoch 10/15, loss: 0.0475
Epoch 15/15, loss: 0.0492
Could not save (no space on device). Model kept in memory — run the next cell to use it.
Done. Run the next cell to load the model for inference.


In [29]:
# Load the fine-tuned model for inference (from disk if saved and valid, else use in-memory model)
import torch
from transformers import AutoConfig
_ft_device = "cuda" if torch.cuda.is_available() else "cpu"
_loaded_from_disk = False
if (FT_OUTPUT_DIR / "pytorch_model.pt").exists():
    try:
        _ft_config = AutoConfig.from_pretrained(str(FT_OUTPUT_DIR))
        _ft_model = AutoModelForCausalLM.from_config(_ft_config)
        _state = torch.load(FT_OUTPUT_DIR / "pytorch_model.pt", map_location="cpu", weights_only=True)
        _ft_model.load_state_dict(_state, strict=True)
        _ft_model = _ft_model.to(_ft_device)
        _ft_tokenizer = AutoTokenizer.from_pretrained(str(FT_OUTPUT_DIR))
        _loaded_from_disk = True
        print("Loaded fine-tuned model from disk.")
    except (RuntimeError, Exception) as _e:
        if "central directory" in str(_e) or "zip" in str(_e).lower():
            print("Saved file is corrupted or truncated. Using in-memory model if available.")
        else:
            print(f"Load failed: {_e}. Using in-memory model if available.")
if not _loaded_from_disk:
    try:
        _ft_model = model_ft.to(_ft_device)
        _ft_tokenizer = tokenizer_ft
        print("Using in-memory model from training cell.")
    except NameError:
        raise RuntimeError("No saved model and no in-memory model. Re-run the training cell above, then this cell.") from None
_ft_model.eval()

def predict_xof_finetuned(text: str, lang: str = "fr", max_new_tokens: int = 15) -> float:
    """Predict price in XOF using the in-notebook fine-tuned open-source model."""
    q = QUESTION_FR if lang == "fr" else QUESTION_EN
    prompt = f"{q}\n\n{text}\n\n{PREFIX_XOF}"
    inputs = _ft_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(_ft_device) for k, v in inputs.items()}
    with torch.no_grad():
        out = _ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=_ft_tokenizer.eos_token_id,
        )
    decoded = _ft_tokenizer.decode(out[0], skip_special_tokens=True)
    continuation = decoded[len(_ft_tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)):].strip()
    return parse_price(continuation)

Saved file is corrupted or truncated. Using in-memory model if available.
Using in-memory model from training cell.


## Predictor — LLM API (stand-in before fine-tuning)

Single call per item; parse numeric answer. Replace with a **fine-tuned** model (e.g. Llama + QLoRA in Colab) for lower cost and better XOF accuracy.

In [30]:
def parse_price(reply: str) -> float:
    if not reply:
        return 0.0
    s = str(reply).replace(",", "").replace(" ", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0

def predict_xof(text: str, lang: str = "fr") -> float:
    q = QUESTION_FR if lang == "fr" else QUESTION_EN
    prompt = f"{q}\n\n{text}\n\n{PREFIX_XOF}"
    try:
        r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0)
        return parse_price(r.choices[0].message.content)
    except Exception as e:
        print(f"API error: {e}")
        return 0.0

## Baselines and evaluation (Day 5 style)

Constant (mean) baseline; compare MAE and R² in XOF.

In [31]:
mean_price = float(np.mean([d["price"] for d in data]))

def baseline_constant(_text):
    return mean_price

def evaluate(data, predictor, n=DEFAULT_N_EVAL):
    subset = data[:n]
    truths = [d["price"] for d in subset]
    preds = [predictor(d["text"]) for d in subset]
    mae = mean_absolute_error(truths, preds)
    r2 = r2_score(truths, preds)
    return mae, r2

mae_base, r2_base = evaluate(data, baseline_constant)
print(f"Constant baseline — MAE: {mae_base:,.0f} XOF  R²: {r2_base:.3f}")

Constant baseline — MAE: 14,619 XOF  R²: 0.000


In [32]:
subset_eval = data[:DEFAULT_N_EVAL]
truths_eval = [d["price"] for d in subset_eval]
preds_llm = [predict_xof(d["text"]) for d in subset_eval]
preds_ft = [predict_xof_finetuned(d["text"]) for d in subset_eval]
mae_llm = mean_absolute_error(truths_eval, preds_llm)
r2_llm = r2_score(truths_eval, preds_llm)
mae_ft = mean_absolute_error(truths_eval, preds_ft)
r2_ft = r2_score(truths_eval, preds_ft)
print(f"LLM (API) — MAE: {mae_llm:,.0f} XOF  R²: {r2_llm:.3f}")
print(f"Fine-tuned (OS) — MAE: {mae_ft:,.0f} XOF  R²: {r2_ft:.3f}")

# Per-item table: baseline, LLM, fine-tuned
n_show = min(8, len(subset_eval))
preds_base_sub = [baseline_constant(d["text"]) for d in subset_eval[:n_show]]
preds_ft_sub = preds_ft[:n_show]
rows = []
for i in range(n_show):
    d = subset_eval[i]
    title = (d["text"].split(chr(10))[0])[:45] + ("." if len(d["text"]) > 45 else "")
    rows.append({"Product": title, "Truth": int(d["price"]), "Baseline": int(preds_base_sub[i]), "LLM": int(preds_llm[i]), "Fine-tuned": int(preds_ft_sub[i]), "|Err| base": int(round(abs(preds_base_sub[i] - d["price"]))), "|Err| LLM": int(round(abs(preds_llm[i] - d["price"]))), "|Err| FT": int(round(abs(preds_ft_sub[i] - d["price"])))})
eval_df = pd.DataFrame(rows)
display(eval_df)

LLM (API) — MAE: 17,793 XOF  R²: -1.193
Fine-tuned (OS) — MAE: 233,335,666,730,000,627,793,920 XOF  R²: -1453379945372412111257220749589210988544.000


,Product,Truth,Baseline,LLM,Fine-tuned,|Err| base,|Err| LLM,|Err| FT
0,"Title: Riz 25 kg (parboiled, local).",12500,12786,25000,12500,287,12500,0
1,Title: Lampe solaire LED 5W.,3500,12786,15000,3500,9287,11500,0
2,Title: Huile de cuisson 1L.,1200,12786,5000,1200,11587,3800,0
3,Title: Ciment 50 kg.,4500,12786,5000,4500,8287,500,0
4,Title: Téléphone Android d'occasion 4G.,35000,12786,150000,3500035000950009349799936,22213,115000,3500035000950009349799936
5,Title: Cartable écolier.,5500,12786,5000,5500,7287,500,0
6,Title: Moustiquaire imprégnée.,2500,12786,15000,2500000,10287,12500,2497500
7,Title: Casque moto.,8500,12786,50000,8500,4287,41500,0


## Results — Model comparison (Week 7 style)

Bar chart of mean absolute error (XOF) for baseline vs LLM, aligned with `results.ipynb`.

In [33]:
results = [
    ("Constant (mean)", "gray", mae_base),
    ("LLM (gpt-4o-mini)", "slateblue", mae_llm),
    ("Fine-tuned (distilgpt2)", "green", mae_ft),
]
labels, colors, values = zip(*results)
fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))
fig.update_layout(
    title="Erreur de prédiction (XOF) — Afrique / Côte d'Ivoire",
    yaxis=dict(title="MAE (XOF)", range=[0, max(values) * 1.15]),
    xaxis=dict(tickangle=-25),
    width=600,
    height=400,
    template="plotly_white",
)
try:
    fig.show()
except ValueError:
    fig.write_html("week7_mae_chart.html")
    print("Chart saved to week7_mae_chart.html (open in browser)")

## Gradio — Compare baseline vs LLM (Week 7 eval)

Pick a sample product or enter a custom description; see **baseline (constant mean)** vs **LLM** prediction. Unlike Week 5 (chat Q&A), this is a **model comparison** demo for price prediction.

In [34]:
import gradio as gr

choices = ["Custom"] + [(d["text"].split(chr(10))[0][:50] + "…") for d in data]

def compare_models(choice: str, custom_text: str) -> str:
    if choice == "Custom":
        text = (custom_text or "").strip()
        if not text:
            return "Enter a product description or pick a sample."
        truth_str = ""
    else:
        idx = choices.index(choice) - 1
        text = data[idx]["text"]
        truth_str = f"Ground truth: {int(data[idx]['price']):,} XOF"
    base = baseline_constant(text)
    llm = predict_xof(text)
    ft = predict_xof_finetuned(text)
    return f"Baseline (mean): {int(base):,} XOF\nLLM: {int(llm):,} XOF\nFine-tuned (OS): {int(ft):,} XOF\n{truth_str}"

with gr.Blocks(title="Baseline vs LLM vs Fine-tuned — XOF") as demo:
    gr.Markdown("**Compare baseline, LLM API, and in-notebook fine-tuned model** for price prediction (XOF). Choose a sample or enter custom description.")
    sel = gr.Dropdown(choices=choices, value=choices[1], label="Sample product")
    custom = gr.Textbox(label="Custom description (used when 'Custom' is selected)", placeholder="e.g. Riz 25 kg, sac parboiled...", lines=3)
    out = gr.Textbox(label="Predictions (XOF)", lines=4)
    sel.change(lambda c: compare_models(c, ""), inputs=[sel], outputs=[out])
    custom.change(lambda c, t: compare_models(c, t), inputs=[sel, custom], outputs=[out])
    demo.load(lambda: compare_models(choices[1], ""), outputs=[out])
demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## Gradio — Price Estimation Pipeline Analytics

Dashboard with **Price Distribution** (histogram), **Market Segments** (pie by price tier in XOF), and **Model Scaling** (learning curve: MAE vs number of evaluation samples). Run the cell below and click **Generate/Refresh Visuals** to update charts.

In [35]:
# Price tiers for XOF (Côte d'Ivoire / West Africa)
def _tier(p):
    if p < 5_000: return "Budget (<5k XOF)"
    if p < 20_000: return "Mid-Range (5k–20k)"
    if p < 50_000: return "Premium (20k–50k)"
    return "Luxury (>50k XOF)"

def build_analytics_plots():
    prices = np.array([d["price"] for d in data])
    n = len(subset_eval)
    # 1) Price distribution histogram
    fig_dist = go.Figure()
    fig_dist.add_trace(go.Histogram(x=prices, nbinsx=min(25, max(10, len(prices)//2)), marker_color="teal", name="Count"))
    fig_dist.update_layout(
        title="Price Distribution in Dataset (XOF)",
        xaxis_title="Price (XOF)",
        yaxis_title="Count",
        template="plotly_white",
        height=400,
        showlegend=False,
    )
    # 2) Market segments pie
    tiers = [_tier(p) for p in prices]
    tier_counts = pd.Series(tiers).value_counts()
    colors = {"Budget (<5k XOF)": "#e74c3c", "Mid-Range (5k–20k)": "#f39c12", "Premium (20k–50k)": "#2ecc71", "Luxury (>50k XOF)": "#3498db"}
    fig_pie = go.Figure(go.Pie(
        labels=tier_counts.index,
        values=tier_counts.values,
        hole=0.4,
        marker_colors=[colors.get(l, "gray") for l in tier_counts.index],
        textinfo="label+percent",
    ))
    fig_pie.update_layout(
        title="Dataset Composition by Price Tier (XOF)",
        template="plotly_white",
        height=400,
    )
    # 3) Learning curve: MAE vs number of eval samples (using existing preds_llm)
    ns = np.arange(1, n + 1, dtype=int)
    mae_at_n = [mean_absolute_error(truths_eval[:k], preds_llm[:k]) for k in ns]
    fig_curve = go.Figure()
    fig_curve.add_trace(go.Scatter(x=ns, y=mae_at_n, mode="lines", name="LLM MAE", line=dict(color="royalblue")))
    fig_curve.add_hline(y=mae_base, line_dash="dash", line_color="red", annotation_text="Baseline (mean)")
    fig_curve.update_layout(
        title="Learning Curve: MAE vs Evaluation Samples (XOF)",
        xaxis_title="Number of Evaluation Examples",
        yaxis_title="Mean Absolute Error (XOF)",
        template="plotly_white",
        height=400,
    )
    return fig_dist, fig_pie, fig_curve

import gradio as gr

with gr.Blocks(title="Price Estimation Pipeline Analytics") as demo_analytics:
    gr.Markdown("# Price Estimation Pipeline Analytics")
    gr.Markdown(f"**Baseline MAE:** {mae_base:,.0f} XOF   **Average Item Price:** {mean_price:,.0f} XOF")
    with gr.Tabs():
        with gr.Tab("Price Distribution"):
            plot_dist = gr.Plot(label="Price Distribution")
        with gr.Tab("Market Segments"):
            plot_pie = gr.Plot(label="Market Segments")
        with gr.Tab("Model Scaling"):
            plot_curve = gr.Plot(label="Model Scaling")
    btn = gr.Button("Generate/Refresh Visuals")

    def refresh():
        return build_analytics_plots()

    btn.click(fn=refresh, outputs=[plot_dist, plot_pie, plot_curve])
    demo_analytics.load(fn=refresh, outputs=[plot_dist, plot_pie, plot_curve])

demo_analytics.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## Gradio — Model Fine-tuning for Price Prediction

Interactive UI: select an open-source model, set training parameters (and optional QLoRA), run **Start Training**, then view **Predictions vs Truth** and metrics (Avg Error in XOF, RMSLE, Hits %). First run the fine-tuning cells above so a saved model exists; this UI can re-train with different settings and refresh the plot.

In [ ]:
import gradio as gr

def _predictions_vs_truth_plot(truths, preds, title="Predictions vs Truth (XOF)"):
    """Build scatter plot: Truth (XOF) vs Guess (XOF) with diagonal line."""
    fig = go.Figure()
    err_pct = np.abs(np.array(preds) - np.array(truths)) / (np.array(truths) + 1e-6)
    colors = ["green" if p < 0.2 else "orange" if p < 0.5 else "red" for p in err_pct]
    fig.add_trace(go.Scatter(x=truths, y=preds, mode="markers", name="Prediction", marker=dict(size=10, color=colors)))
    mx = max(max(truths), max(preds), 1) * 1.05
    fig.add_trace(go.Scatter(x=[0, mx], y=[0, mx], mode="lines", line=dict(dash="dash", color="gray"), name="Ideal"))
    fig.update_layout(
        title=title,
        xaxis_title="Truth (XOF)",
        yaxis_title="Guess (XOF)",
        template="plotly_white",
        height=400,
        showlegend=True,
    )
    return fig

def _metrics_str(truths, preds):
    truths_np, preds_np = np.array(truths, dtype=float), np.array(preds, dtype=float)
    mae = float(np.mean(np.abs(preds_np - truths_np)))
    rmsle = float(np.sqrt(np.mean((np.log1p(preds_np) - np.log1p(truths_np)) ** 2)))
    hits = float(np.mean(np.abs(preds_np - truths_np) / (truths_np + 1e-6) < 0.2)) * 100
    return f"Avg Error: {mae:,.0f} XOF | RMSLE: {rmsle:.2f} | Hits (within 20%): {hits:.1f}%"

def run_ft_ui(model_name, lr, epochs, batch_size, use_qlora, lora_rank, lora_alpha, compare_baselines):
    """Run fine-tuning with selected params and return (status, plot, metrics). Uses existing train_data and subset_eval."""
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from datasets import Dataset
        from torch.utils.data import DataLoader
        out_dir = Path("./pricer_xof_ui_finetuned")
        out_dir.mkdir(parents=True, exist_ok=True)
        texts = [r["prompt"] + " " + str(r["completion"]) for r in train_data]
        ds = Dataset.from_dict({"text": texts})
        tok = AutoTokenizer.from_pretrained(model_name)
        tok.pad_token = tok.eos_token
        def _tok(ex):
            return tok(ex["text"], truncation=True, max_length=256, padding="max_length")
        ds = ds.map(_tok, batched=True, remove_columns=["text"])
        ds.set_format("torch")
        model = AutoModelForCausalLM.from_pretrained(model_name)
        dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(dev)
        loader = DataLoader(ds, batch_size=int(batch_size), shuffle=True)
        opt = torch.optim.AdamW(model.parameters(), lr=float(lr))
        model.train()
        for _ in range(int(epochs)):
            for batch in loader:
                input_ids = batch["input_ids"].to(dev)
                attention_mask = batch["attention_mask"].to(dev)
                labels = input_ids.clone()
                labels[attention_mask == 0] = -100
                opt.zero_grad()
                out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                out.loss.backward()
                opt.step()
        model.save_pretrained(str(out_dir))
        tok.save_pretrained(str(out_dir))
        # Inference on subset_eval
        model.eval()
        dev = next(model.parameters()).device
        preds = []
        for d in subset_eval:
            q = QUESTION_FR
            prompt = f"{q}\n\n{d['text']}\n\n{PREFIX_XOF}"
            inp = tok(prompt, return_tensors="pt", truncation=True, max_length=256)
            inp = {k: v.to(dev) for k, v in inp.items()}
            with torch.no_grad():
                out = model.generate(**inp, max_new_tokens=15, do_sample=False, pad_token_id=tok.eos_token_id)
            dec = tok.decode(out[0], skip_special_tokens=True)
            cont = dec[len(tok.decode(inp["input_ids"][0], skip_special_tokens=True)):].strip()
            preds.append(parse_price(cont))
        truths = [d["price"] for d in subset_eval]
        fig = _predictions_vs_truth_plot(truths, preds)
        metrics = _metrics_str(truths, preds)
        if compare_baselines:
            metrics += f"  |  Baseline MAE: {mae_base:,.0f} XOF  |  LLM MAE: {mae_llm:,.0f} XOF"
        return "Training finished.", fig, metrics
    except Exception as e:
        import traceback
        return f"Error: {e}\n{traceback.format_exc()}", None, "—"

# Initial plot/metrics from existing fine-tuned model (preds_ft)
try:
    _initial_fig = _predictions_vs_truth_plot(truths_eval, preds_ft)
    _initial_metrics = _metrics_str(truths_eval, preds_ft)
except NameError:
    _initial_fig = None
    _initial_metrics = "Run the fine-tuning and evaluation cells above first."

with gr.Blocks(title="Model Fine-tuning for Price Prediction", theme=gr.themes.Soft()) as demo_ft:
    gr.Markdown("# Model Fine-tuning for Price Prediction")
    with gr.Row():
        with gr.Column(scale=1):
            model_dd = gr.Dropdown(
                choices=["distilgpt2", "gpt2"],
                value="distilgpt2",
                label="Select Model",
                info="Small open-source models (no HF token).",
            )
            lr_num = gr.Number(value=0.0002, label="Learning Rate", minimum=1e-5, maximum=0.01)
            epochs_sl = gr.Slider(1, 10, value=3, step=1, label="Epochs")
            batch_sl = gr.Slider(1, 8, value=2, step=1, label="Batch Size")
            gr.Markdown("**QLoRA (GPU only)**")
            use_qlora = gr.Checkbox(value=False, label="Enable QLoRA")
            lora_rank = gr.Slider(2, 32, value=8, step=1, label="LoRA Rank")
            lora_alpha = gr.Slider(8, 64, value=16, step=1, label="LoRA Alpha")
            compare_cb = gr.Checkbox(value=True, label="Compare with Baselines")
            train_btn = gr.Button("Start Training", variant="primary")
            status_tb = gr.Textbox(label="Status", lines=3, interactive=False)
        with gr.Column(scale=1):
            with gr.Tabs():
                with gr.Tab("Model Performance"):
                    perf_plot = gr.Plot(label="Predictions vs Truth")
                    metrics_tb = gr.Markdown(_initial_metrics)

    def on_train(model_name, lr, epochs, batch_size, use_qlora, lora_rank, lora_alpha, compare_baselines):
        status, fig, metrics = run_ft_ui(model_name, lr, epochs, batch_size, use_qlora, lora_rank, lora_alpha, compare_baselines)
        return status, fig, metrics

    train_btn.click(
        fn=on_train,
        inputs=[model_dd, lr_num, epochs_sl, batch_sl, use_qlora, lora_rank, lora_alpha, compare_cb],
        outputs=[status_tb, perf_plot, metrics_tb],
    )
    demo_ft.load(fn=lambda: ("Ready. Click Start Training to fine-tune.", _initial_fig, _initial_metrics), outputs=[status_tb, perf_plot, metrics_tb])

demo_ft.launch()